# Foco-LLM: Qwen 2.5 7B com treino estendido

Experimento exploratório em uma GPU T4: baseline 7B, QLoRA NF4 por 640 passos, avaliação dos checkpoints 320 e 640 nos mesmos 40 casos congelados. Nenhum gabarito de avaliação entra no treino. A comparação 7B/3B mistura tamanho do modelo e outras diferenças de arquitetura; a comparação 320/640 no 7B isola a duração do ajuste. O checkpoint de 320 é diagnóstico, não um critério para escolher o modelo pelo teste.

Ative GPU T4 x2 e Internet antes de executar. O experimento usa a primeira T4. Salve uma versão com Save & Run All para preservar os artefatos. Modelo: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct (Apache 2.0).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This locked project requires Python 3.12; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
repository = Path("/kaggle/working/foco-llm")
if not repository.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "feat/task-balanced-training",
            "--single-branch",
            "https://github.com/Mathwesm/foco-llm.git",
            str(repository),
        ],
        check=True,
    )
subprocess.run(["git", "pull", "--ff-only"], cwd=repository, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repository, check=True)
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)

## 1. Preparar e medir o 7B sem ajuste

A base de treino é reproduzida e validada pelo SHA-256 já fixado no projeto.

In [ ]:
output = Path("/kaggle/working/foco-output-7b")
command = ["poetry", "run", "python", "scripts/run_kaggle_selection.py"]
for mode in ("prepare", "baseline"):
    subprocess.run(
        [*command, mode, "--model-size", "7b", "--output", str(output)],
        cwd=repository,
        check=True,
    )

## 2. Treinar por 640 passos

Checkpoint íntegro a cada 80 passos. O ajuste pode ser retomado na mesma pasta se a sessão cair.

In [ ]:
subprocess.run(
    [
        *command,
        "train",
        "--model-size",
        "7b",
        "--output",
        str(output),
        "--steps",
        "640",
        "--checkpoint-every",
        "80",
    ],
    cwd=repository,
    check=True,
)
training_runs = sorted((output / "training").glob("*/summary.json"))
if len(training_runs) != 1:
    raise RuntimeError(f"Expected one completed training run, found {len(training_runs)}")
run_directory = training_runs[0].parent
for step in (320, 640):
    if not (run_directory / f"step-{step:04d}" / "complete.json").is_file():
        raise RuntimeError(f"Missing complete adapter checkpoint at step {step}")

## 3. Avaliar os checkpoints sem alterar o conjunto de teste

Os 40 casos são os mesmos usados para avaliar o 3B. Inspecione principalmente `similar-4`, mas registre todas as condições e respostas brutas.

In [ ]:
import json

summaries = {}
baseline_files = sorted((output / "baseline").glob("*/summary.json"))
if len(baseline_files) != 1:
    raise RuntimeError(f"Expected one baseline summary, found {len(baseline_files)}")
summaries["baseline"] = json.loads(baseline_files[0].read_text(encoding="utf-8"))
for step in (320, 640):
    adapter = run_directory / f"step-{step:04d}"
    subprocess.run(
        [
            *command,
            "adapted",
            "--model-size",
            "7b",
            "--output",
            str(output),
            "--adapter",
            str(adapter),
        ],
        cwd=repository,
        check=True,
    )
    matches = []
    for manifest_path in (output / "adapted").glob("*/manifest.json"):
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest["runtime"].get("adapter_step") == str(step):
            matches.append(manifest_path.parent / "summary.json")
    if len(matches) != 1:
        raise RuntimeError(f"Expected one adapted summary at step {step}, found {len(matches)}")
    summaries[f"step-{step:04d}"] = json.loads(matches[0].read_text(encoding="utf-8"))
for name, summary in summaries.items():
    print(name, summary["groups"]["similar-4"])

## 4. Visualizar e guardar as métricas

O gráfico e o JSON são salvos em `/kaggle/working/foco-output-7b`.

In [ ]:
import matplotlib.pyplot as plt

forms = ("text-1", "text-2", "text-4", "similar-4")
labels = ("baseline", "step-0320", "step-0640")
colors = ("#0072B2", "#E69F00", "#009E73")
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = list(range(len(forms)))
for offset, name, color in zip((-0.26, 0.0, 0.26), labels, colors, strict=True):
    values = [summaries[name]["groups"][form]["selection_exact"] for form in forms]
    ax.bar(
        [position + offset for position in positions], values, width=0.26, label=name, color=color
    )
ax.set_xticks(positions, forms)
ax.set_ylim(0, 10)
ax.set_ylabel("Exact selections (of 10 cases)")
ax.set_xlabel("Diagnostic condition")
ax.set_title("Qwen 2.5 7B: evidence selection by training duration")
ax.legend()
fig.tight_layout()
fig.savefig(output / "selection-comparison-7b.png", dpi=160, bbox_inches="tight")
(output / "comparison-summary.json").write_text(json.dumps(summaries, indent=2), encoding="utf-8")
plt.close(fig)